# event_log 변환 SQL 생성 (ad_exposure 추가분 전용)

`ad_exposure_logs2.sql` (generate_ad_exposure_logs2.ipynb 실행 결과)을 읽어,
`event_log` 테이블에 맞는 INSERT문으로 변환합니다.

## event_log 테이블 컬럼
```
id, user_id, event_name, event_timestamp, product_id, product_name, product_category,
approved_amount, action_type, coupon_code, discount_amount, expiry_date,
search_keyword, page_name, dwell_time, review_rating, earned_points, earn_reason,
login_id, created_at, ad_id, client_uuid
```

## 매핑 규칙 (ad_exposure)
- user_id, login_id, client_uuid, event_timestamp (공통)
- product_id, product_name, product_category, ad_id
- 그 외 컬럼은 모두 NULL (id는 SERIAL이므로 INSERT문에 포함하지 않음)

## 사용 방법
1. `ad_exposure_logs2.sql` 파일을 이 노트북과 같은 디렉토리에 둔다
2. 노트북을 실행하면 `event_log_insert_ad_exposure2.sql`이 생성된다

In [10]:
import re
import json
import os

In [11]:
# ───────────────────────────────────────────
# 설정값
# ───────────────────────────────────────────
SOURCE_SQL_FILE = 'ad_exposure_logs2.sql'
OUTPUT_SQL_FILE = 'event_log_insert_ad_exposure2.sql'

In [12]:
# event_log 테이블 컬럼 순서 (id 제외)
EVENT_LOG_COLUMNS = [
    'user_id', 'event_name', 'event_timestamp', 'product_id', 'product_name',
    'product_category', 'approved_amount', 'action_type', 'coupon_code',
    'discount_amount', 'expiry_date', 'search_keyword', 'page_name',
    'dwell_time', 'review_rating', 'earned_points', 'earn_reason',
    'login_id', 'created_at', 'ad_id', 'client_uuid'
]

In [13]:
def extract_rows_from_sql(sql_text):
    """
    INSERT INTO ... (history_timestamp, json_log) VALUES
      ('2026-06-01 00:00:01.000000', '{"event_name": ...}'),
      ...
    형식의 SQL 문자열에서 (history_timestamp, json_log) 튜플 리스트를 추출한다.
    json_log 내부의 '' (escaped single quote)는 '로 복원한다.
    """
    pattern = re.compile(
        r"\(\s*'([^']*(?:''[^']*)*)'\s*,\s*'((?:[^']|'')*)'\s*\)",
        re.DOTALL
    )

    rows = []
    for m in pattern.finditer(sql_text):
        history_ts_raw = m.group(1)
        json_log_raw   = m.group(2)

        history_ts = history_ts_raw.replace("''", "'")
        json_log   = json_log_raw.replace("''", "'")

        rows.append((history_ts, json_log))

    return rows

In [14]:
def sql_literal(value):
    """파이썬 값을 SQL 리터럴 문자열로 변환 (None -> NULL, 문자열은 quote+escape)"""
    if value is None:
        return 'NULL'
    if isinstance(value, bool):
        return 'TRUE' if value else 'FALSE'
    if isinstance(value, (int, float)):
        return str(value)
    escaped = str(value).replace("'", "''")
    return f"'{escaped}'"

In [15]:
def map_to_event_log(history_ts, json_log):
    """
    history_timestamp + json_log(dict)를 event_log 테이블 컬럼 dict로 매핑한다.
    이 노트북은 ad_exposure 전용이므로 product 정보 + ad_id만 채운다.
    """
    data = json.loads(json_log)

    row = {col: None for col in EVENT_LOG_COLUMNS}

    # ── 공통 ──
    row['user_id']         = data.get('user_id')
    row['event_name']      = data.get('event_name')
    row['event_timestamp'] = data.get('event_timestamp')
    row['login_id']        = data.get('user_login_id')
    row['created_at']      = history_ts
    row['client_uuid']     = data.get('client_uuid')

    # ── ad_exposure 전용 ──
    row['product_id']       = data.get('productId')
    row['product_name']     = data.get('productName')
    row['product_category'] = data.get('productCategory')
    row['ad_id']            = data.get('adId')

    return row

In [16]:
if not os.path.exists(SOURCE_SQL_FILE):
    raise FileNotFoundError(f'{SOURCE_SQL_FILE} 이 없습니다. generate_ad_exposure_logs2.ipynb를 먼저 실행해 주세요.')

with open(SOURCE_SQL_FILE, 'r', encoding='utf-8') as f:
    sql_text = f.read()

raw_rows = extract_rows_from_sql(sql_text)
all_event_rows = [map_to_event_log(history_ts, json_log) for history_ts, json_log in raw_rows]

print(f'✅ {SOURCE_SQL_FILE} → {len(all_event_rows)}건 변환')

✅ ad_exposure_logs2.sql → 1000건 변환


In [17]:
# event_log INSERT SQL 생성
columns_str = ', '.join(EVENT_LOG_COLUMNS)

lines  = [f'INSERT INTO event_log ({columns_str}) VALUES']
values = []

for row in all_event_rows:
    literals = [sql_literal(row[col]) for col in EVENT_LOG_COLUMNS]
    values.append('  (' + ', '.join(literals) + ')')

lines.append(',\n'.join(values) + ';')
event_log_sql = '\n'.join(lines)

with open(OUTPUT_SQL_FILE, 'w', encoding='utf-8') as f:
    f.write(event_log_sql)

print(f'✅ {len(all_event_rows)}건 event_log INSERT SQL 생성 완료 → {OUTPUT_SQL_FILE}')

✅ 1000건 event_log INSERT SQL 생성 완료 → event_log_insert_ad_exposure2.sql


In [18]:
# ── 미리보기 ──
print('=== EVENT_LOG INSERT SQL (앞 1000자) ===')
print(event_log_sql[:1000])

=== EVENT_LOG INSERT SQL (앞 1000자) ===
INSERT INTO event_log (user_id, event_name, event_timestamp, product_id, product_name, product_category, approved_amount, action_type, coupon_code, discount_amount, expiry_date, search_keyword, page_name, dwell_time, review_rating, earned_points, earn_reason, login_id, created_at, ad_id, client_uuid) VALUES
  (40, 'ad_exposure', '2025-10-24T13:57:05.000+09:00', '1107', '선반 수납선반 팬트리 모듈 다용도 미니 벽 선반장 시스템 조립 600 5단', '가구/인테리어', NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'user0040', '2025-10-24 13:57:06.000000', 100, 'a529b23f-d033-44e9-a5cb-d3153e40977a'),
  (93, 'ad_exposure', '2026-05-07T17:14:43.000+09:00', '1517', '호카 본디 9 1162011', '패션잡화', NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'user0093', '2026-05-07 17:14:44.000000', 13, '5bdeaa21-ac40-46b6-a9ed-7a178836568a'),
  (18, 'ad_exposure', '2026-04-29T13:51:50.000+09:00', '1916', '삼성전자 갤럭시 S21 공기계 SM-G991N 256GB 휴대폰,알뜰폰', '디지털/가전', NULL, NULL, NULL, NU